In [11]:
import json
import pandas as pd
import numpy as np
import json
import os
import sys
!pip install folium
sys.path.append(os.path.abspath(".."))
from src.visualization import generate_dispatch_map

Defaulting to user installation because normal site-packages is not writeable


In [12]:
data_dir = os.path.join('..', 'data', 'processed')
df = pd.read_csv(os.path.join(data_dir, 'cleaned_delivery_orders.csv'))
matrices = np.load(os.path.join(data_dir, 'vrp_mathematical_matrices.npz'), allow_pickle=True)

with open('mip_results.json', 'r') as f: mip = json.load(f)
with open('ga_results.json', 'r') as f: ga = json.load(f)

In [13]:
sys.path.append(os.path.abspath(".."))
from src.ga_engine import run_ga_engine, slice_matrices

with open('mip_scalability_results.json', 'r') as f:
    mip_scalability = json.load(f)

matched_rows = []
for mip_row in mip_scalability:
    N = mip_row['N']
    sliced = slice_matrices(matrices, N) if 'matrices' in dir() else None
    if sliced is None:
        matrices = np.load(os.path.join('..', 'data', 'processed', 'vrp_mathematical_matrices.npz'), allow_pickle=True)
        sliced = slice_matrices(matrices, N)

    ga_row = run_ga_engine(
        n_nodes=N + 1, population_size=100, penalty_value=10000,
        vehicle_capacity=matrices['vehicle_capacities'].tolist(),
        vehicle_shifts_min=matrices['vehicle_shifts_min'].tolist(),
        matrices=sliced, df=df.iloc[:N + 1], mx_generations=200, patience=20,
        mutation_rate=0.05, crossover_rate=0.8, seed=42, verbose=False
    )
    matched_rows.append({
        'N (orders)': N,
        'MIP Revenue (LKR)': mip_row['Revenue'],
        'MIP Time (s)': round(mip_row['Execution_Time'], 2),
        'MIP Status': mip_row['Status'] + (' [time-limited]' if mip_row['Time_Limited'] else ''),
        'GA Revenue (LKR)': ga_row['actual_revenue'],
        'GA Time (s)': round(ga_row['execution_time'], 2),
    })

matched_df = pd.DataFrame(matched_rows)
display(matched_df)

with open('ga_results.json', 'r') as f:
    ga_full = json.load(f)

print(f"\nFull-scale reference (all 192 orders - MIP is intractable at this size):")
print(f"GA Revenue: LKR {ga_full['Revenue']:,.0f} | GA Fitness: {ga_full['Fitness_Score']:,.0f} | "
      f"GA Time: {ga_full['Execution_Time']:.2f}s")


,N (orders),MIP Revenue (LKR),MIP Time (s),MIP Status,GA Revenue (LKR),GA Time (s)
0,5,2175400.0,12.08,Optimal,201100.0,0.14
1,10,2315400.0,119.77,Optimal [time-limited],329000.0,0.22
2,15,2524300.0,119.29,Optimal,496500.0,0.26
3,20,1727700.0,118.01,Optimal,658800.0,0.26



Full-scale reference (all 192 orders - MIP is intractable at this size):
GA Revenue: LKR 2,314,900 | GA Fitness: 2,254,900 | GA Time: 125.32s


In [14]:
# load saved GA results
with open('ga_results.json', 'r') as f:
    ga_results = json.load(f)
    
best_routes = ga_results['Best Routes']
deferred_orders = ga_results['Deferred Orders']

print("Number of deferred orders:", len(deferred_orders))
print("Total Routes Dispatched:", len([r for r in best_routes if len(r) > 2]))

# lat and long
data_path = os.path.join("..", "data", "processed", "cleaned_delivery_orders.csv")

# generate map
dispatch_map = generate_dispatch_map(data_path, best_routes)

# save map to HTML
html_file = "dispatch_map.html"
dispatch_map.save(html_file)

print(f"Dispatch map saved to {html_file}. You can open this file in a web browser to view the routes.")

Number of deferred orders: 135
Total Routes Dispatched: 21
Dispatch map saved to dispatch_map.html. You can open this file in a web browser to view the routes.
